# Solución · Lab 07 · Arrays

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crapdev/sparkLab/blob/main/labs/07-arrays/solution.ipynb)

**Objetivo:** Trabajar con columnas que contienen listas: crearlas con split, acceder por índice, medirlas con size, desplegarlas con explode y consultarlas con array_contains.

**Tiempo estimado:** 35 minutos

> Esta es la solución completa. Intenta resolver el laboratorio por tu cuenta antes
> de leerla: equivocarse y corregir es donde más se aprende.

## Preparación

Esta celda instala la versión de PySpark con la que se probó el laboratorio y comprueba
que hay una versión de Java compatible. En Colab tarda alrededor de un minuto la primera vez.

In [ ]:
import importlib.metadata
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.2.0"
MIN_JAVA_VERSION = 17


def installed_pyspark_version():
    try:
        return importlib.metadata.version("pyspark")
    except importlib.metadata.PackageNotFoundError:
        return None


def java_major_version():
    java = shutil.which("java")
    if java is None:
        return 0
    output = subprocess.run([java, "-version"], capture_output=True, text=True).stderr
    match = re.search(r'version "(\d+)', output)
    return int(match.group(1)) if match else 0


if installed_pyspark_version() != PYSPARK_VERSION:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )

if java_major_version() < MIN_JAVA_VERSION:
    if "google.colab" not in sys.modules:
        raise RuntimeError("Spark 4 necesita Java 17 o superior. Instálalo y repite esta celda.")
    subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print(f"PySpark {PYSPARK_VERSION} y Java {java_major_version()} listos.")

La siguiente celda define `check`, la función que usan los tests. No hace falta que la
modifiques: ejecútala y sigue.

In [ ]:
RESULTS = {}


def check(name, condition, hint):
    """Run one test: `condition` is a function that must return True."""
    try:
        passed = bool(condition())
        error = None
    except Exception as exc:  # A broken answer is a failed test, not a crash.
        passed, error = False, exc
    RESULTS[name] = passed
    if passed:
        print(f"✅ {name}")
        return
    detail = f" ({type(error).__name__}: {error})" if error else ""
    print(f"❌ {name}{detail}")
    print(f"   Pista: {hint}")
    assert passed, f"{name} · {hint}"

## Sesión de Spark

Esta celda crea la `SparkSession` del laboratorio e importa las funciones como `F`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master("local[*]").appName("lab-07-arrays").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} listo.")

## Datos

Esta celda descarga `bigmart_sales.csv` la primera vez y los deja en la carpeta `data/`.
Usa `dataset_path("nombre")` para obtener la ruta de un archivo.

In [ ]:
import os
import urllib.request

DATA_BASE_URLS = [
    "https://raw.githubusercontent.com/crapdev/sparkLab/main/public/data",
]
DATA_DIR = os.environ.get("PYSPARK_LABS_DATA_DIR", "data")
DATASETS = ["bigmart_sales.csv"]


def dataset_path(name):
    """Return the local path of a dataset, downloading it the first time."""
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        return path
    os.makedirs(DATA_DIR, exist_ok=True)
    for base_url in DATA_BASE_URLS:
        try:
            urllib.request.urlretrieve(f"{base_url}/{name}", path)
            return path
        except OSError:
            continue  # Try the next mirror.
    raise RuntimeError(f"No se pudo descargar {name}. Revisa tu conexión y repite la celda.")


for dataset in DATASETS:
    print("Listo:", dataset_path(dataset))

In [ ]:
sales = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(dataset_path("bigmart_sales.csv"))
)

sales.groupBy("Outlet_Type").count().orderBy("Outlet_Type").show(truncate=False)

## Tarea 1 · De texto a lista con `split`

`F.split(columna, separador)` parte un texto por el separador y devuelve un **array**: una
columna cuyo valor es una lista. `Outlet_Type` tiene valores como `Supermarket Type1`, dos
palabras separadas por un espacio.

Crea `parts` añadiendo a `sales` la columna `type_parts` con `Outlet_Type` partido por el
espacio.

In [ ]:
parts = sales.withColumn("type_parts", F.split("Outlet_Type", " "))

In [ ]:
check(
    "Tarea 1 · type_parts es un array de strings",
    lambda: dict(parts.dtypes).get("type_parts") == "array<string>",
    'Usa F.split("Outlet_Type", " "). El resultado tiene que ser una columna nueva llamada type_parts.',
)
check(
    "Tarea 1 · todas las listas tienen 2 elementos",
    lambda: parts.filter(F.size("type_parts") == 2).count() == 8523,
    'El separador es un espacio, " ". Con otro separador la lista tendría un solo elemento.',
)
check(
    "Tarea 1 · Outlet_Type sigue siendo texto",
    lambda: dict(parts.dtypes)["Outlet_Type"] == "string" and len(parts.columns) == 13,
    "Añade type_parts como columna nueva; no sobrescribas Outlet_Type.",
)

Mira cómo se ve un array en `show()` y en el esquema:

In [ ]:
parts.select("Outlet_Type", "type_parts").distinct().orderBy("Outlet_Type").show(truncate=False)
parts.select("type_parts").printSchema()

## Tarea 2 · Un elemento por su posición

Para sacar un elemento de un array se usa el índice entre corchetes, **empezando en 0**:
`F.col("type_parts")[0]` es la primera palabra y `[1]` la segunda. El resultado es una
columna normal del tipo de los elementos.

Crea `coded` añadiendo a `parts` la columna `type_code` con la **segunda** palabra de
`type_parts` (`Type1`, `Type2`, `Type3` o `Store`).

In [ ]:
coded = parts.withColumn("type_code", F.col("type_parts")[1])

In [ ]:
check(
    "Tarea 2 · type_code es string",
    lambda: dict(coded.dtypes).get("type_code") == "string",
    'Usa F.col("type_parts")[1]. Un elemento de un array<string> es un string.',
)
check(
    "Tarea 2 · 5577 filas con Type1 y 1083 con Store",
    lambda: coded.filter(F.col("type_code") == "Type1").count() == 5577 and coded.filter(F.col("type_code") == "Store").count() == 1083,
    "La segunda palabra es el índice 1, no el 2. El índice 0 daría Supermarket o Grocery.",
)

Pedir un índice que no existe **no devuelve nulo**: en Spark 4 falla al ejecutar, como
el `cast` imposible. Esta celda pide el sexto elemento de una lista de dos:

In [ ]:
try:
    parts.select(F.col("type_parts")[5]).show(1)
except Exception as error:
    print(type(error).__name__)
    print(str(error).split("\n")[0])

Si de verdad quieres `NULL` cuando el índice se pasa, el propio mensaje lo dice: `F.get`.

In [ ]:
parts.select(F.get("type_parts", 5).alias("sixth")).show(1)

## Tarea 3 · Cuántos elementos tiene la lista

`F.size(array)` devuelve el número de elementos. Combinada con `split`, cuenta palabras.

`Item_Type` tiene nombres de una, dos y tres palabras (`Dairy`, `Soft Drinks`,
`Fruits and Vegetables`). Crea `counted` añadiendo a `sales` la columna `word_count` con
el número de palabras de `Item_Type`.

In [ ]:
counted = sales.withColumn("word_count", F.size(F.split("Item_Type", " ")))

In [ ]:
check(
    "Tarea 3 · word_count es un entero",
    lambda: dict(counted.dtypes).get("word_count") == "int",
    'Usa F.size(F.split("Item_Type", " ")). size devuelve un int.',
)
check(
    "Tarea 3 · 3260 de una palabra, 3511 de dos y 1752 de tres",
    lambda: counted.filter(F.col("word_count") == 1).count() == 3260
    and counted.filter(F.col("word_count") == 2).count() == 3511
    and counted.filter(F.col("word_count") == 3).count() == 1752,
    'Parte Item_Type por el espacio, " ", y mide la lista con F.size.',
)

In [ ]:
counted.select("Item_Type", "word_count").distinct().orderBy(F.desc("word_count"), "Item_Type").show(5, truncate=False)

## Tarea 4 · Una fila por elemento con `explode`

`F.explode(array)` genera **una fila por cada elemento** de la lista. Las demás columnas
se repiten en cada fila nueva. Es la forma de pasar de "una fila con una lista" a "una
fila por valor", que es lo que necesitas para agrupar o contar por elemento.

Crea `exploded` añadiendo a `parts` la columna `type_word` con cada palabra de
`type_parts` en su propia fila.

In [ ]:
exploded = parts.withColumn("type_word", F.explode("type_parts"))

In [ ]:
check(
    "Tarea 4 · 17046 filas: el doble",
    lambda: exploded.count() == 17046,
    'Usa F.explode("type_parts"). Cada fila tiene 2 palabras, así que salen 2 filas por cada una de las 8523.',
)
check(
    "Tarea 4 · type_word es string y las demás columnas siguen ahí",
    lambda: dict(exploded.dtypes).get("type_word") == "string" and len(exploded.columns) == 14,
    "explode no quita columnas: añade type_word a las 13 que tenía parts.",
)
check(
    "Tarea 4 · la palabra Supermarket aparece 7440 veces",
    lambda: exploded.filter(F.col("type_word") == "Supermarket").count() == 7440,
    "Supermarket está en 5577 + 928 + 935 filas de sales; tras explode, en una fila por cada una.",
)

In [ ]:
exploded.select("Item_Identifier", "Outlet_Type", "type_word").show(4, truncate=False)
exploded.groupBy("type_word").count().orderBy("type_word").show()

`explode` solo acepta arrays (y mapas). Sobre la columna de texto original falla con un
mensaje que conviene reconocer:

In [ ]:
try:
    sales.withColumn("type_word", F.explode("Outlet_Type")).show(1)
except Exception as error:
    print(type(error).__name__)
    print(str(error).split(";")[0])

## Tarea 5 · ¿Está en la lista? `array_contains`

`F.array_contains(array, valor)` devuelve `true` si el valor está en la lista y `false` si
no. Como todas las comparaciones de texto en Spark, distingue mayúsculas.

Crea `flagged` añadiendo a `parts` la columna `type1_flag`: `true` cuando `type_parts`
contiene `Type1`.

In [ ]:
flagged = parts.withColumn("type1_flag", F.array_contains("type_parts", "Type1"))

In [ ]:
check(
    "Tarea 5 · type1_flag es boolean",
    lambda: dict(flagged.dtypes).get("type1_flag") == "boolean",
    'Usa F.array_contains("type_parts", "Type1"). Devuelve una columna boolean.',
)
check(
    "Tarea 5 · 5577 true y 2946 false",
    lambda: flagged.filter(F.col("type1_flag")).count() == 5577 and flagged.filter(~F.col("type1_flag")).count() == 2946,
    'El valor buscado es exactamente "Type1", con mayúscula. Con "type1" todo saldría false.',
)

Como `explode`, `array_contains` exige un array. Aplicarla a `Outlet_Type`, que es texto,
da un error de tipos:

In [ ]:
try:
    sales.withColumn("type1_flag", F.array_contains("Outlet_Type", "Type1")).show(1)
except Exception as error:
    print(type(error).__name__)
    print(str(error).split(";")[0])

## Tarea 6 · Listas vacías y nulas: `explode_outer`

`explode` tiene una consecuencia que a veces no se quiere: una fila cuya lista está
**vacía** o es **nula** desaparece, porque no hay elementos que desplegar.
`F.explode_outer` conserva esas filas con un `NULL` en la columna nueva.

El DataFrame `tagged` tiene tres productos: uno con dos etiquetas, uno sin ninguna y uno
con la lista nula.

In [ ]:
tagged = spark.createDataFrame(
    [(1, ["bio", "local"]), (2, []), (3, None)],
    "product_id INT, tags ARRAY<STRING>",
)
tagged.show()

Crea `one_per_tag` con `explode`, y `all_products` con `explode_outer`, las dos con la
columna nueva `tag`.

In [ ]:
one_per_tag = tagged.withColumn("tag", F.explode("tags"))
all_products = tagged.withColumn("tag", F.explode_outer("tags"))

In [ ]:
check(
    "Tarea 6 · explode deja 2 filas, solo del producto 1",
    lambda: one_per_tag.count() == 2 and one_per_tag.filter(F.col("product_id") != 1).count() == 0,
    'Usa F.explode("tags"): los productos sin etiquetas desaparecen.',
)
check(
    "Tarea 6 · explode_outer deja 4 filas y conserva los productos 2 y 3",
    lambda: all_products.count() == 4 and all_products.filter(F.col("tag").isNull()).count() == 2,
    'Usa F.explode_outer("tags"): los productos 2 y 3 quedan con tag en NULL.',
)

In [ ]:
one_per_tag.show()
all_products.show()

## Para terminar · el separador de `split` es una expresión regular

El segundo argumento de `split` no es un texto literal sino una **regex**, igual que en
`regexp_replace`. Con un espacio no se nota, pero con un punto o una barra vertical sí:
`.` significa "cualquier carácter" y `|` significa "o". Hay que escaparlos con `\\`.

In [ ]:
versions = spark.createDataFrame([("4.2.0",), ("3.5.1",)], "version STRING")
versions.select(
    "version",
    F.split("version", ".").alias("wrong"),
    F.split("version", "\\.").alias("right"),
    F.split("version", "\\.")[0].alias("major"),
).show(truncate=False)

## Resumen

Ejecuta esta celda al final. Si todos los tests pasan, vuelve al sitio y marca el
laboratorio como completado.

In [ ]:
EXPECTED_TESTS = 14
passed_tests = sum(RESULTS.values())

print(f"{passed_tests} de {EXPECTED_TESTS} tests superados")
for test_name, passed in RESULTS.items():
    print("  ✅" if passed else "  ❌", test_name)

assert passed_tests == EXPECTED_TESTS, "Todavía quedan tareas por resolver."
print("\n🎉 ¡Laboratorio completado! Vuelve al sitio y márcalo como hecho.")